In [1]:
import time
from datetime import datetime
import pandas as pd
import re

import torch
import torch.nn as nn
import torch.nn.functional as actF

import torch_geometric.nn as pyg_nn
import torch_geometric.utils as pyg_utils

import networkx as nx
import numpy as np
import torch.optim as optim

import torch_geometric.transforms as T

from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

/home/pedro/miniconda3/envs/envml/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
import sys
from pathlib import Path
print("cwd:", os.getcwd())
print("sys.path[0]:", sys.path[0])

project_root = Path.cwd().parent
sys.path.append(str(project_root))

cwd: /home/pedro/Documents/Trabajo/TSCF/slc-gnns/gnns
sys.path[0]: /home/pedro/miniconda3/envs/envml/lib/python312.zip


In [3]:
data_folder = "../data/input_data/"

df_data_all = pd.read_parquet(data_folder+"SLC_MAP_all_data_2011_2022.parquet")
df_data_all = df_data_all.drop(columns=["[Florida (Puente Ruta 5)] Nivel de agua (h) [m]"]) # Drop nivel (= Caudal)
df_data_all

,[Florida (Puente Ruta 5)] Caudal (Q) [m3/s],[INIA Las Brujas] Evapotranspiracion Penman (ET) [mm],[INIA Las Brujas] Heliofania (Hel) [hs],[INIA Las Brujas] Humedad relativa media (HR) [%],[INIA Las Brujas] Radiacion solar (RS) [cal/cm2/dia],[INIA Las Brujas] Temperatura del aire media (TA) [ºC(24hs)],[INIA Las Brujas] Temperatura del aire maxima (TA) [ºC],[INIA Las Brujas] Temperatura del aire minima (TA) [ºC],[INIA Las Brujas] Velocidad del viento (VV) [2m/km/24h],[PAM Station] MAP (P) [mm]
date,,,,,,,,,,
2011-01-01,0.347,5.9,9.9,70.0,576.7,22.6,28.2,17.5,205.0,0.00
2011-01-02,0.333,5.9,9.3,72.0,555.6,23.2,29.4,18.4,223.4,0.60
2011-01-03,0.353,6.6,9.8,61.0,572.6,23.8,29.6,19.9,216.0,0.00
2011-01-04,0.383,3.9,2.2,72.0,308.1,23.0,28.2,18.9,165.9,0.00
2011-01-05,0.383,4.1,5.2,77.0,412.1,23.4,30.3,19.6,80.5,0.00
...,...,...,...,...,...,...,...,...,...,...
2022-03-24,0.084,2.5,5.9,74.0,329.9,18.5,23.2,11.8,85.1,3.56
2022-03-25,0.062,1.5,2.8,92.0,239.1,13.9,18.6,11.2,112.2,5.14
2022-03-26,0.057,1.7,2.2,89.0,220.3,17.4,21.7,12.7,113.8,1.14


In [4]:
target_station = "Florida (Puente Ruta 5)"
target_variable = "Caudal (Q) [m3/s]"

all_cols = list(df_data_all.columns)
all_stations = [
    re.search(r"\[(.*?)\]", e).group(1)
    for e in all_cols
]
station_order = list(dict.fromkeys(all_stations)) # Remove duplicates from list while preserving order of elements

stations_coords = pd.read_csv(data_folder+"other_data/SL_stations.csv")

stations_coords = stations_coords[stations_coords["station"].isin(station_order)] # Keep only stations inside Florida's basin
stations_coords = stations_coords.set_index('station') # Define new index

# --- (1) Se definen las distancias ---

coords_list = [
        (stations_coords.loc[s, 'longitud'],
        stations_coords.loc[s, 'latitud'])
    for s in station_order
]

target_station_coords = [stations_coords.loc[target_station, 'longitud'],
                            stations_coords.loc[target_station, 'latitud']]

In [5]:
# --- (2) Se define la conectividad del grafo ---

edge_index = torch.tensor([
    [1, 2],
    [0, 0]
], dtype=torch.long)
edge_index

tensor([[1, 2],
        [0, 0]])

In [6]:
def IDW_Calc(d, p=2):
    d = np.array(d, dtype=float)
    
    weights = 1 / (d ** p)
    weights /= weights.sum()
    
    return weights

In [7]:
def calc_weights(func):
    row, col = edge_index

    coords = torch.tensor(coords_list, dtype=torch.float)

    distances = torch.linalg.norm(
        coords[row] - coords[col],
        dim=1
    )

    # ---- Prevent extremely small distances ----
    min_dist = distances.quantile(0.1)  # 10th percentile
    distances = torch.clamp(distances, min=min_dist)

    if func=="IDW":


        # ---- Gaussian weighting ----
        weights = torch.tensor(IDW_Calc(distances), dtype=torch.float32)

    elif func=="dist":

        # Use distances directly as weights
        weights = distances

        # Optional normalization to [0,1]
        weights = (weights - weights.min()) / (
            weights.max() - weights.min()
        )


    elif func=="gauss":

        # ---- Gaussian weighting ----
        sigma = distances.mean()

        weights = torch.exp(
            - (distances ** 2) / (2 * sigma ** 2)
        )

        # Normalize to [0,1]
        weights = (weights - weights.min()) / (
            weights.max() - weights.min()
        )

    else:
        print(f"WRONG FUNC: {func}")
        return 0

    dist_weight = weights.view(-1, 1)


    return dist_weight

In [8]:
from collections import defaultdict

def extract_station(col):
    return re.match(r"\[(.*?)\]", col).group(1)

station_columns = defaultdict(list)

for col in df_data_all.columns:
    st = extract_station(col)
    station_columns[st].append(col)

In [9]:
def extract_variable(col):
    return re.sub(r"\[.*?\]\s*", "", col)

variable_order = [c.split("] ", 1)[1] for c in df_data_all.columns]
variable_order = list(dict.fromkeys(variable_order))

In [10]:
from sklearn.model_selection import train_test_split

train_ratio = 0.8
test_ratio = 0.2

# First split: train vs (val + test)
df_train, df_test = train_test_split(
    df_data_all,
    train_size=train_ratio,
    shuffle=False
)

# Second split: val vs test
# dataset_val, dataset_test = train_test_split(
#     df_data_all,
#     train_size=val_ratio / (1 - train_ratio),
#     shuffle=False
# )
#print(f"N° val samples: {len(dataset_val.index)}")

print(f"N° train samples: {len(df_train.index)}")
print(f"N° test samples: {len(df_test.index)}")


N° train samples: 3284
N° test samples: 821


In [24]:
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

scalerX = MinMaxScaler()
scalerY = MinMaxScaler()

target_col = f"[{target_station}] {target_variable}"

df_train_scaled = df_train.copy()

X_train = df_train_scaled.drop(columns=[target_col])
y_train = df_train_scaled[[target_col]] 

# Fit scalers
X_train_scaled = scalerX.fit_transform(X_train)
y_train_scaled = scalerY.fit_transform(y_train)

# Assign back
df_train_scaled[X_train.columns] = X_train_scaled
df_train_scaled[target_col] = y_train_scaled

df_test_scaled = df_test.copy()

X_test = df_test_scaled.drop(columns=[target_col])
y_test = df_test_scaled[[target_col]]

# Transform only
X_test_scaled = scalerX.transform(X_test)
y_test_scaled = scalerY.transform(y_test)

# Assign back
df_test_scaled[X_test.columns] = X_test_scaled
df_test_scaled[target_col] = y_test_scaled

In [12]:
import torch
import numpy as np

num_nodes = len(station_order)
num_features = len(variable_order)
num_time_train = len(df_train_scaled)
num_time_test = len(df_test_scaled)

X_all_train = torch.zeros(
    (num_nodes, num_time_train, num_features),
    dtype=torch.float32
)

X_all_test = torch.zeros(
    (num_nodes, num_time_test, num_features),
    dtype=torch.float32
)

for i, station in enumerate(station_order):

    for j, variable in enumerate(variable_order):

        col_name = f"[{station}] {variable}"

        if col_name in df_train_scaled.columns:

            X_all_train[i, :, j] = torch.tensor(
                df_train_scaled[col_name].values,
                dtype=torch.float32
            )

            X_all_test[i, :, j] = torch.tensor(
                df_test_scaled[col_name].values,
                dtype=torch.float32
            )

        else:
            # missing variable at station
            X_all_train[i, :, j] = 0.0
            X_all_test[i, :, j] = 0.0

In [13]:
import torch
from torch_geometric.data import Data

# Return list of windowed graph data
# In this case: 
# - input_window = "window"
# - output_window = 1

def build_windowed_graphs(
    x_all,                     # [num_nodes, T, num_features]
    edge_index,                # [2, num_edges]
    window,
    target_feature_idx,        # feature to predict
    target_node_idx,           # None → predict for all nodes
    edge_weight=None,          # [num_nodes, num_features]
    use_streamflow_as_input=False
):

    num_nodes, T, num_features = x_all.shape

    data_list = []

    for t in range(T - window):

        # Input window
        x_window = x_all[:, t:t+window, :].clone()  # shape: [num_nodes, window, num_features]

        if not use_streamflow_as_input:
            x_window[:, :, target_feature_idx] = 0 # Don't use target as input (set it to zero)

        # Target timestep
        y_full = x_all[:, t+window, target_feature_idx]  # shape: [num_nodes]
        
        # If: 
        #   target_node_idx = None --> Predict all variables
        #   target_node_idx = idx --> Predict 'idx' variable
        if target_node_idx is not None:
            y = y_full[target_node_idx]  # shape: scalar
        else:
            y = y_full  # shape: [num_nodes]

        data = Data(
            x=x_window,
            edge_index=edge_index,
            edge_weight=edge_weight,
            y=y
        )

        data_list.append(data)

    return data_list

In [14]:
target_feature_idx = variable_order.index(target_variable) # Variable --> Feature
target_node_idx = station_order.index(target_station) # Station --> Node

dist_weight = calc_weights("IDW")

edge_weight = dist_weight/dist_weight.sum() # Distance normalization --> edge weights
window_size = 36

use_streamflow_as_input=False

dataset_train = build_windowed_graphs(
    X_all_train,
    edge_index,
    window=window_size,
    target_feature_idx=target_feature_idx,
    target_node_idx=target_node_idx,
    edge_weight=edge_weight,
    use_streamflow_as_input=use_streamflow_as_input
)

dataset_test = build_windowed_graphs(
    X_all_test,
    edge_index,
    window=window_size,
    target_feature_idx=target_feature_idx,
    target_node_idx=target_node_idx,
    edge_weight=edge_weight,
    use_streamflow_as_input=use_streamflow_as_input
)

print(f"Length train: {len(dataset_train)}")
print(f"Graph train: {dataset_train[0]}")
print(f"Length test: {len(dataset_test)}")
print(f"Graph test: {dataset_test[0]}")

Length train: 3248
Graph train: Data(x=[3, 36, 10], edge_index=[2, 2], y=4.089299181941897e-05, edge_weight=[2, 1])
Length test: 785
Graph test: Data(x=[3, 36, 10], edge_index=[2, 2], y=0.0004632504715118557, edge_weight=[2, 1])


### GNN Model Definition

In [15]:
from source.models import RecursiveAttentionGNN

In [16]:
from torch_geometric.loader import DataLoader
batch_dize = 64

train_loader = DataLoader(dataset_train, batch_size=batch_dize, shuffle=False)
test_loader = DataLoader(dataset_test, batch_size=batch_dize, shuffle=False)
from sklearn.metrics import r2_score as nse_metric

def train(model, criterion, optimizer, train_loader, use_weigths):
    model.train()
    total_train_loss = 0

    all_preds_inv = []
    all_targets_inv = []

    for data in train_loader:

        optimizer.zero_grad()  # Clear gradients.

        if use_weigths:
            out = model(data.x, data.edge_index, data.edge_weight)  # Perform a single forward pass.
        else:
            out = model(data.x, data.edge_index)  # Perform a single forward pass.

        # out: [num_nodes x batch_size, num_features] ---> We want 'Caudal' for Node 'Florida' (slice array by num_nodes)
        target_prediction = out[target_node_idx::num_nodes]

        train_loss = criterion(target_prediction, data.y)  # Compute the loss solely based on the training nodes.

        train_loss.backward()  # Derive gradients.
        optimizer.step()  # Update parameters based on gradients.

        total_train_loss += train_loss.item()

        y_pred = target_prediction.detach().cpu().numpy().reshape(-1, 1)
        y_true = data.y.detach().cpu().numpy().reshape(-1, 1)

        all_preds_inv.append(scalerY.inverse_transform(y_pred))
        all_targets_inv.append(scalerY.inverse_transform(y_true))
    
    all_preds_inv = np.concatenate(all_preds_inv, axis=0).ravel()
    all_targets_inv = np.concatenate(all_targets_inv, axis=0).ravel()

    # Before appending inverse pred for NSE evaluation, make negative predictions zero
    all_preds_inv[all_preds_inv<0] = 0

    nse = nse_metric(all_targets_inv, all_preds_inv)

    avg_loss = total_train_loss / len(train_loader)

    return avg_loss, nse

def test(model, criterion, test_loader, use_weigths):
    model.eval()

    total_loss = 0.0

    all_preds_inv = []
    all_targets_inv = []

    with torch.no_grad():
        for data in test_loader:

            if use_weigths:
                out = model(data.x, data.edge_index, data.edge_weight)  # Perform a single forward pass.
            else:
                out = model(data.x, data.edge_index)  # Perform a single forward pass.

            # out: [num_nodes x batch_size, num_features] ---> We want 'Caudal' for Node 'Florida' (slice array by num_nodes)
            target_prediction = out[target_node_idx::num_nodes]

            target_prediction_inv = scalerY.inverse_transform(target_prediction.reshape(-1, 1))
            target_true_inv = scalerY.inverse_transform(data.y.reshape(-1, 1))

            # Loss (e.g., MSELoss)
            loss = criterion(target_prediction, data.y)

            total_loss += loss.item()

            # Before appending inverse pred for NSE evaluation, make negative predictions zero
            target_prediction_inv[target_prediction_inv<0] = 0

            all_preds_inv.append(torch.from_numpy(target_prediction_inv))
            all_targets_inv.append(torch.from_numpy(target_true_inv))

    all_preds_inv = torch.cat(all_preds_inv)

    all_targets_inv = torch.cat(all_targets_inv)

    nse = nse_metric(all_targets_inv, all_preds_inv)

    avg_loss = total_loss / len(test_loader)

    return avg_loss, nse

In [17]:
from source.models import TemporalAttentionGNN

In [18]:
edge_weight

tensor([[0.2048],
        [0.7952]])

In [19]:
num_features = dataset_train[0]["x"].shape[2]

model = TemporalAttentionGNN(
    num_features=num_features,
    gnn_hidden=16,
    out_channels=1
)

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)
criterion = nn.MSELoss()

num_epochs = 200

train_loss_list = []
test_loss_list = []
test_nse_list = []
train_nse_list = []

use_weigths = False

for epoch in range(num_epochs):
    loss, train_nse = train(model, criterion, optimizer, train_loader, use_weigths)
    test_loss, test_nse = test(model, criterion, test_loader, use_weigths)

    train_loss_list.append(loss)
    test_loss_list.append(test_loss)
    test_nse_list.append(test_nse)
    train_nse_list.append(train_nse)

    print(
        f"Epoch [{1+epoch:03d}/{num_epochs}] | "
        f"Train Loss: {loss:.6f} | "
        f"Train NSE: {train_nse:.2f} | "
        f"Test Loss: {test_loss:.6f} | "
        f"Test NSE: {test_nse:.2f}"
    )

TypeError: TemporalAttentionGNN.__init__() missing 2 required positional arguments: 'concat' and 'use_weights'